In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))
from src.adapter.api.template.comment import SpecialAbilitiesComment

In [2]:
from pathlib import Path

folder_path = Path("../raw_data/Ho_ba_hoc_sinh/Mot1/")

excel_files = [str(f) for f in folder_path.glob("*.xlsx")]

print(excel_files)

['..\\raw_data\\Ho_ba_hoc_sinh\\Mot1\\HOC_BA_11_VOMINHKHANG_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\Mot1\\HOC_BA_12_PHAMNHATDANGKHOA_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\Mot1\\HOC_BA_13_TRANHOANGKY_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\Mot1\\HOC_BA_14_NGUYENBINHPHUONGLAM_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\Mot1\\HOC_BA_15_THAIPHUONGLINH_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\Mot1\\HOC_BA_16_HONGUYENKHANHNGAN_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\Mot1\\HOC_BA_17_NGUYENNGOCKIMNGAN_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\Mot1\\HOC_BA_18_LUUGIANGHI_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\Mot1\\HOC_BA_19_NGUYENBAONGOC_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\Mot1\\HOC_BA_1_TRUONGPHUCAN_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\Mot1\\HOC_BA_20_NGUYENPHAMANNHIEN_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\Mot1\\HOC_BA_21_PHAMANNHIEN_NAM

In [3]:
from src.adapter.database.postgres_repository import PostgresStudentRepository
from src.adapter.database.postges_manager import postgres_manager
student_repo = PostgresStudentRepository(postgres_manager.session)
(student_repo.exists({"name": 'Võ Minh Khang'})).code

2026-04-16 23:17:23,836 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-04-16 23:17:23,839 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-04-16 23:17:24,109 INFO sqlalchemy.engine.Engine select current_schema()
2026-04-16 23:17:24,112 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-04-16 23:17:24,386 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-04-16 23:17:24,389 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-04-16 23:17:24,664 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-16 23:17:24,681 INFO sqlalchemy.engine.Engine SELECT student.code AS student_code, student.name AS student_name, student.card_id AS student_card_id, student.edu_id AS student_edu_id, student.date_of_birth AS student_date_of_birth, student.gender AS student_gender, student.ethnicity AS student_ethnicity, student.status AS student_status, student.phone AS student_phone, student.nationality AS student_nationality, student.address AS student_address, student.place_of_birth

'20252026.01.Mo1.011'

In [4]:
from src.adapter.database.postges_manager import postgres_manager
from src.adapter.database.mongo_manager import mongo_manager
from src.adapter.graph.neo4j_manager import neo4j_manager
from src.application.core import SystemCore

session = postgres_manager.session
db = mongo_manager.get_metadata_db()
manager = neo4j_manager
repo = SystemCore(session, db, manager)

d:\Work\Iu\hoc-ba-so\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3325.22it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import re
from openpyxl import load_workbook
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional
import json
import re

def normalize_text(value: object) -> str:
    if value is None:
        return ""
    text = str(value).replace("\n", " ").strip()
    text = re.sub(r"\s+", " ", text)
    return text


def extract_student_name(raw_text: str) -> str:
    text = normalize_text(raw_text)
    if ":" in text:
        return text.split(":", 1)[1].strip()
    return text


def find_header_row_and_comment_col(ws) -> tuple[int, int]:
    """
    Tìm dòng header có:
    - cột tên môn học
    - cột nhận xét
    """
    for row_idx in range(1, ws.max_row + 1):
        row_values = [normalize_text(ws.cell(row_idx, col_idx).value) for col_idx in range(1, ws.max_column + 1)]

        subject_col = None
        comment_col = None

        for col_idx, value in enumerate(row_values, start=1):
            value_lower = value.lower()
            if "môn học" in value_lower and "hoạt động giáo dục" in value_lower:
                subject_col = col_idx
            if "nhận xét" in value_lower:
                comment_col = col_idx

        if subject_col and comment_col:
            return row_idx, comment_col

    raise ValueError("Không tìm thấy dòng header hoặc cột 'Nhận xét' trong sheet.")


def extract_comments_from_sheet(
    file_path: str | Path,
    sheet_name: str = "MH Mot 1 (2024-2025)",
):
    wb = load_workbook(file_path, data_only=True)
    if sheet_name not in wb.sheetnames:
        raise ValueError(f"Không tìm thấy sheet: {sheet_name}")

    ws = wb[sheet_name]

    # Tên học sinh thường ở A1: "Họ và tên học sinh: Trương Phúc An"
    student_name = extract_student_name(ws["A1"].value)

    header_row, comment_col = find_header_row_and_comment_col(ws)
    subject_col = 1  # trong file này môn học nằm ở cột A

    result = SpecialAbilitiesComment(code=(student_repo.exists({"name": student_name})).code)

    subject_to_field = {
        "tiếng việt": "vietnamese_comment",
        "toán": "mathematics_comment",
        "th-cn (tin học)": "informatics_comment",
        "tin học": "informatics_comment",
        "khoa học": "science_comment",
        "lịch sử và địa lí": "history_and_geography_comment",
        "ngoại ngữ tiếng anh": "english_comment",
        "tiếng anh": "english_comment",
        "công nghệ": "technology_comment",
        "nghệ thuật (âm nhạc)": "music_comment",
        "âm nhạc": "music_comment",
        "nghệ thuật (mĩ thuật)": "arts_comment",
        "mĩ thuật": "arts_comment",
        "mỹ thuật": "arts_comment",
        "đạo đức": "civics_comment",
        "giáo dục thể chất": "physical_education_comment",
        "hoạt động trải nghiệm": "experiential_activities_comment",
    }

    for row_idx in range(header_row + 1, ws.max_row + 1):
        subject = normalize_text(ws.cell(row_idx, subject_col).value)
        comment = normalize_text(ws.cell(row_idx, comment_col).value)

        if not subject:
            continue

        subject_key = subject.lower()

        for pattern, field_name in subject_to_field.items():
            if pattern in subject_key:
                setattr(result, field_name, comment or None)
                break

    return result


if __name__ == "__main__":
    for file_path in excel_files:
        data = extract_comments_from_sheet(file_path, sheet_name="MH Mot 1 (2024-2025)")
        break

await repo.add_student_comment_special(data)

2026-04-16 23:17:54,344 INFO sqlalchemy.engine.Engine SELECT student.code AS student_code, student.name AS student_name, student.card_id AS student_card_id, student.edu_id AS student_edu_id, student.date_of_birth AS student_date_of_birth, student.gender AS student_gender, student.ethnicity AS student_ethnicity, student.status AS student_status, student.phone AS student_phone, student.nationality AS student_nationality, student.address AS student_address, student.place_of_birth AS student_place_of_birth 
FROM student 
WHERE student.name = %(name_1)s 
 LIMIT %(param_1)s
2026-04-16 23:17:54,346 INFO sqlalchemy.engine.Engine [cached since 29.67s ago] {'name_1': 'Võ Minh Khang', 'param_1': 1}
2026-04-16 23:17:54,490 INFO sqlalchemy.engine.Engine SELECT student.code AS student_code, student.name AS student_name, student.card_id AS student_card_id, student.edu_id AS student_edu_id, student.date_of_birth AS student_date_of_birth, student.gender AS student_gender, student.ethnicity AS student_e

{'student_code': '20252026.01.Mo1.011',
 'vietnamese': {'evidence': 'em chưa đọc được bài',
  'confident': 0.5571904182434082,
  'indicator': 'Biết đọc thầm.',
  'full_comment': 'Em chưa đọc được bài, chưa hoàn thành nội dung môn học.'},
 'mathematics': {'evidence': 'chưa hoàn thành nội dung môn học',
  'confident': 0.4389886260032654,
  'indicator': 'Củng cố và hoàn thiện các kĩ năng: rút gọn được phân số.',
  'full_comment': 'Em chưa có kỹ năng tính, chưa hoàn thành nội dung môn học.'},
 'informatics': {'evidence': 'em còn lúng túng khi thực hiện thao tác vẽ đường gấp khúc khép kín',
  'confident': 0.5698985457420349,
  'indicator': 'Cầm chuột đúng cách.',
  'full_comment': 'Em còn lúng túng khi thực hiện thao tác vẽ đường gấp khúc khép kín.'},
 'english': {'evidence': 'em hoàn thành nội dung môn học',
  'confident': 0.5126744508743286,
  'indicator': 'Đọc những đoạn văn bản ngắn, đơn giản đã được chuẩn bị trước về những chủ đề quen thuộc như bạn bè, gia đình, nhà trường, ….',
  'ful